## Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

* Tracking agent behavior with logging, analytics, and debugging.
* Transforming prompts, tool selection, and output formatting.
* Adding retries, fallbacks, and early termination logic.
* Applying rate limits, guardrails, and PII detection.


In [36]:
import os
from langchain_groq import ChatGroq

model = ChatGroq(
    model = "llama-3.3-70b-versatile"
)

## Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

* Long-running conversations that exceed context windows.
* Multi-turn dialogues with extensive history.
* Applications where preserving full conversation context matters.


In [37]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

agents = create_agent(
    model = model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model = model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)


In [38]:
"""run with thread"""
config = {
    "configurable" : {
        "thread_id" : "test_1"
    }
}

In [39]:
questions = [
    "what is 4*12?",
    "What is 8*9?",
    "Where is Burj Khalifa?",
    "Founder of OLA?",
    "Which is the Most Widely used Web Framework in the world?",
    "In which year did the inagural duronto express run and between which two cities?"
]

In [40]:
for q in questions:
    response = agents.invoke(
        {
            "messages":[HumanMessage(content=q)]
        }, config=config
    )
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='what is 4*12?', additional_kwargs={}, response_metadata={}, id='0866b09e-0a36-4111-95d5-2e49a6ddba13'), AIMessage(content='4 ** 12 = 48.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.043854824, 'completion_tokens_details': None, 'prompt_time': 0.00219619, 'prompt_tokens_details': None, 'queue_time': 0.055315409, 'total_time': 0.046051014}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fd22d-dc24-7342-8d0d-4bfa37f8ceba-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='what is 4*12?', additional_kwargs={}, response_metadata={}, id='0866b09e-0a36-4111-95d5-2e49a6ddba13'), AI

### Token Size MiddleWare

In [41]:
# from langchain.agents import create_agent
# from langchain.agents.middleware import SummarizationMiddleware
# from langchain_core.tools import tool
# from langchain_core.messages import HumanMessage
# from langgraph.checkpoint.memory import InMemorySaver

# @tool
# def search_hotels(city: str) -> str:
#     """Search hotels - returns long response to use more tokens."""
#     return f"""Hotels in {city}:
# 1. Grand Hotel - 5 star, $350/night, spa, pool, gym
# 2. City Inn - 4 star, $180/night, business center
# 3. Budget Stay - 3 star, $75/night, free wifi"""

# agents = create_agent(
#     model = model,
#     tools=[search_hotels],
#     checkpointer=InMemorySaver(),
#     middleware=[
#         SummarizationMiddleware(
#             model = model,
#             trigger=("tokens", 200),
#             keep=("tokens", 100)
#         )
#     ]
# )

# def count_tokens(messages):
#     total_chars = sum(len(str(m.content)) for m in messages)
#     return total_chars // 4

# """run with thread"""
# config = {
#     "configurable" : {
#         "thread_id" : "test_1"
#     }
# }

In [42]:

# cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

# for city in cities:
#     response = agents.invoke(
#         {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
#         config=config
#     )

#     tokens = count_tokens(response["messages"])
#     print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
#     print(f"{response['messages']}")

### **Human In the Loop MiddleWare**

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:

* High-stakes operations requiring human approval (e.g. database writes, financial transactions).
* Compliance workflows where human oversight is mandatory.
* Long-running conversations where human feedback guides the agent.


In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [44]:
def read_email_tools(email:str)->str:
    """mock function to connect email by an ID"""
    return f"Email connect for ID {email}"

def send_email_tool(rep: str, subject: str, body: str)->str:
    """mock function to send an email."""
    return f"Email sent {rep} with subjct '{subject}'"

In [45]:
model = ChatGroq(
     model = "openai/gpt-oss-120b"
)

In [46]:
agents = create_agent(
    model=model, 
    tools = [read_email_tools, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions" : ["approve","edit", "reject"]
                },
                "read_email_tools":False
            }
        )
    ]
)

In [47]:
config = {
    "configurable" : {
        "thread_id" : "test-approve"
    }
}

result = agents.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send email to karfa@gmail.com with subject 'Hello' and body 'How are you?'"
            )
        ]
    },
    config=config
)

In [48]:
result

{'messages': [HumanMessage(content="Send email to karfa@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='87dadf89-a9a8-4267-9003-e6ec26cb98cc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters: body, rep (maybe reply-to?), subject. The user wants to send email to karfa@gmail.com. The tool signature: send_email_tool takes body, rep, subject. "rep" maybe recipient? So set rep to "karfa@gmail.com". We\'ll call function.', 'tool_calls': [{'id': 'fc_ce46a343-a06a-4d86-888d-669eeacbd185', 'function': {'arguments': '{"body":"How are you?","rep":"karfa@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 173, 'total_tokens': 290, 'completion_time': 0.244307174, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.007519437, 'prompt_to

In [51]:
from langgraph.types import Command
if "__interrupt__" in result:
    print("Pause! approving...")

    result = agents.invoke(
        Command(
            resume = {
                "decisions": [
                    {
                        "type" : "edit",
                        "edited_action" : {
                            "name" : "send_email_tool",
                            "args": {
                                "rep" : "riku@gmail.com",
                                "subject" : "greeting",
                                "body" : "Congratulations!"
                            }
                        }
                    }
                ]
            }
        ),
        config=config
    )

result

Pause! approving...


{'messages': [HumanMessage(content="Send email to karfa@gmail.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='87dadf89-a9a8-4267-9003-e6ec26cb98cc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to send email using send_email_tool. Provide parameters: body, rep (maybe reply-to?), subject. The user wants to send email to karfa@gmail.com. The tool signature: send_email_tool takes body, rep, subject. "rep" maybe recipient? So set rep to "karfa@gmail.com". We\'ll call function.', 'tool_calls': [{'id': 'fc_ce46a343-a06a-4d86-888d-669eeacbd185', 'function': {'arguments': '{"body":"How are you?","rep":"karfa@gmail.com","subject":"Hello"}', 'name': 'send_email_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 117, 'prompt_tokens': 173, 'total_tokens': 290, 'completion_time': 0.244307174, 'completion_tokens_details': {'reasoning_tokens': 70}, 'prompt_time': 0.007519437, 'prompt_to